# H1/H2/H3 grid — Conformal Burden v2 (cheap-first last-layer arms), then **STOP**

Pre-registered campaign v2, spec §3/§6. Runs the **last-layer arms** (ERM, DFR, AFR,
last-layer-GroupDRO, balanced subsampling) on **Waterbirds + CelebA**, on **two frozen backbones**
(ERM-ResNet-50 primary + CLIP ViT-B/32 secondary), **3 training seeds × 10 calibration splits**,
**APS/RAPS/THR**, **full ρ sweep {0.95,0.9,0.8,0.7,0.6,0.5}**.

**Reports (spec §8):** H1 on the **accuracy-matched** divergence (primary; raw labeled
"uncontrolled"), H2 ranking inversion, H3 shift survival → `RESULTS_study.md` + CSVs + figures.

**Then STOPS** before the optional heavy **full-GroupDRO fine-tune** and any 3rd/4th dataset
(cheap-first checkpoint). Every arm carries the §2 gates; an arm below its worst-group-accuracy
floor is **excluded with a reason** (logged to BLOCKERS), never shipped as valid.

Run top-to-bottom on a **GPU** runtime.

## 0. Parameters — **EDIT THESE**

In [ ]:
# ===================== EDIT THESE =====================
REPO_SOURCE   = "git"
REPO_URL      = "https://github.com/<YOUR_USER>/vgscp.git"   # EDIT
REPO_BRANCH   = "main"
REPO_DRIVE_ZIP= "/content/drive/MyDrive/vgscp.zip"
DRIVE_CACHE   = "/content/drive/MyDrive/vgscp_cache"

SEEDS         = 3      # training seeds (spec: >=3)
N_SPLITS      = 10     # calibration/test splits (spec: >=10)
RESNET_EPOCHS = 10     # ERM ResNet-50 training epochs per dataset

WATERBIRDS_URL = "https://nlp.stanford.edu/data/dro/waterbird_complete95_forest2water2.tar.gz"
# CelebA: provide the extracted dataset yourself (list_attr_celeba.txt + list_eval_partition.txt +
# img_align_celeba/). Set CELEBA_DRIVE to its folder on Drive, or leave "" to SKIP CelebA (the run
# proceeds Waterbirds-only and logs CelebA as skipped — never silently mixed).
CELEBA_DRIVE   = ""    # e.g. "/content/drive/MyDrive/celeba"
# ======================================================
import os, sys, time, subprocess
def sh(cmd, **kw):
    print("$", cmd); return subprocess.run(cmd, shell=True, **kw)

## 1. GPU + install

In [ ]:
import torch
print("CUDA:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")
subprocess.run("pip -q install open_clip_torch ftfy regex tqdm pyyaml scikit-learn scipy pandas matplotlib torchvision", shell=True)

## 2. Mount Drive + get repo

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
os.makedirs(DRIVE_CACHE, exist_ok=True)
REPO_DIR = "/content/vgscp"
if REPO_SOURCE == "git":
    sh(f"rm -rf {REPO_DIR} && git clone --branch {REPO_BRANCH} {REPO_URL} {REPO_DIR}")
else:
    sh(f"rm -rf {REPO_DIR} && mkdir -p {REPO_DIR} && unzip -q {REPO_DRIVE_ZIP} -d {REPO_DIR}")
os.chdir(REPO_DIR); sys.path.insert(0, REPO_DIR)
print("repo:", os.getcwd())

## 3. Datasets + env vars (Waterbirds required; CelebA optional)

In [ ]:
def fetch(url, name, to):
    os.makedirs(to, exist_ok=True)
    tb = os.path.join(DRIVE_CACHE, name)
    if not os.path.exists(tb): sh(f"wget -q -O '{tb}' '{url}'")
    sh(f"tar -xzf '{tb}' -C '{to}'"); return to

fetch(WATERBIRDS_URL, "waterbirds.tar.gz", "/content/data/waterbirds")
os.environ["WATERBIRDS_ROOT"] = "/content/data/waterbirds"

CELEBA_OK = bool(CELEBA_DRIVE) and os.path.isdir(CELEBA_DRIVE)
if CELEBA_OK:
    os.environ["CELEBA_ROOT"] = CELEBA_DRIVE
    print("CelebA root:", CELEBA_DRIVE)
else:
    print("[note] CELEBA_DRIVE not set / not found -> CelebA SKIPPED (Waterbirds-only run).")

# persist feature caches to Drive
for c in ("cache_clip", "cache_resnet"):
    sh(f"rm -rf results/{c}"); os.makedirs(f"{DRIVE_CACHE}/{c}", exist_ok=True); os.makedirs("results", exist_ok=True)
    sh(f"ln -s {DRIVE_CACHE}/{c} results/{c}")
print("WATERBIRDS_ROOT=", os.environ.get("WATERBIRDS_ROOT"))

## 4. Build GridData (extract + cache features) for each (dataset × backbone)
Triggers backbone feature extraction: CLIP ViT-B/32 (cached) and an **ERM ResNet-50** trained
in-domain per dataset (cached). This is the one-time backbone cost — NOT the optional heavy
full-GroupDRO fine-tune (that is post-checkpoint).

In [ ]:
from study_robust_train.datasets import build_griddata

def cfg_for(dataset):
    base = {"clip": {"model_name": "ViT-B-32", "pretrained": "openai", "device": "cuda",
                     "cache_dir": "results/cache_clip"},
            "resnet": {"device": "cuda", "epochs": RESNET_EPOCHS, "lr": 1e-3, "batch_size": 128,
                       "cache_dir": "results/cache_resnet"}}
    if dataset == "waterbirds":
        base["dataset"] = {"root": os.environ["WATERBIRDS_ROOT"], "image_size": 224,
                           "n_classes": 2, "download": False}
    else:
        base["dataset"] = {"root": os.environ["CELEBA_ROOT"], "n_classes": 2}
    return base

KEYS = [("waterbirds", "resnet50_erm"), ("waterbirds", "clip_vitb32")]
if CELEBA_OK:
    KEYS += [("celeba", "resnet50_erm"), ("celeba", "clip_vitb32")]

data, skipped = {}, []
for ds, bb in KEYS:
    try:
        t = time.time()
        gd = build_griddata(ds, bb, cfg_for(ds), seed=0)
        data[(bb, ds)] = gd
        print(f"[built] {bb}/{ds}: train {gd.train[0].shape}, eval {gd.eval_domain[0].shape} "
              f"({(time.time()-t)/60:.1f} min)")
    except Exception as e:
        skipped.append((bb, ds, str(e)))
        print(f"[SKIP] {bb}/{ds}: {e}")
print("built keys:", list(data.keys()), "| skipped:", [(b,d) for b,d,_ in skipped])

## 5. Run the grid (5 methods × 3 seeds × 3 scores × 6 ρ × 10 splits) + verdicts

In [ ]:
from study_robust_train.grid import run_grid, write_csv, write_results_md
from study_robust_train.figures import make_figures

t = time.time()
out = run_grid(data, seeds=tuple(range(SEEDS)), n_splits=N_SPLITS)   # full RHO_SWEEP + APS/RAPS/THR
print(f"[grid] {len(out['records'])} records, {len(out['excluded'])} excluded "
      f"({(time.time()-t)/60:.1f} min)")

os.makedirs("results/study", exist_ok=True)
write_csv(out["records"], "results/study/grid_records.csv")
write_results_md(out, "RESULTS_study.md", synthetic=False)
figs = make_figures(out, "results/study/figures")
print("wrote RESULTS_study.md, results/study/grid_records.csv,", len(figs), "figures")

## 6. Verdict summary + §2 gate check (worst-group accuracy per arm)

In [ ]:
import numpy as np
# §2 worst-group accuracy per (key, method) — DFR on ERM-ResNet should land ~0.86-0.92 on Waterbirds
print("=== worst-group accuracy per arm (§2 gate; head property) ===")
for (bb, ds) in data:
    for m in sorted({r["method"] for r in out["records"] if (r["backbone"],r["dataset"])==(bb,ds)}):
        wg = np.mean([r["worst_group_acc"] for r in out["records"]
                      if (r["backbone"],r["dataset"])==(bb,ds) and r["method"]==m])
        print(f"  {bb}/{ds:11s} {m:18s} worst-group acc = {wg:.3f}")
if out["excluded"]:
    print("\n=== EXCLUDED arms (below floor) ===")
    for e in out["excluded"]:
        print(f"  {e['backbone']}/{e['dataset']} {e['method']} seed{e['seed']}: {e['worst_group_acc']:.3f} < {e['floor']}")

print("\n=== H1 / H2 / H3 ===")
for key, v in out["verdicts"].items():
    if "note" in v: print(key, "->", v["note"]); continue
    h1go = {m: mr["GO"] for m, mr in v["h1"]["methods"].items()}
    print(f"{key}: H1 GO={h1go} | H2 inversion={v['h2']['inversion']} "
          f"(acc-top={v['h2']['top_by_accuracy']}, burden-top={v['h2']['top_by_burden']})")

## 7. Show RESULTS_study.md + figures

In [ ]:
from IPython.display import Image, Markdown, display
display(Markdown(open("RESULTS_study.md", encoding="utf-8").read()))
for p in figs:
    display(Image(p))

## 8. STOP — cheap-first arms complete
The last-layer arms on Waterbirds + CelebA are done and reported. **STOP for review.** Do NOT
proceed to the optional heavy **full-GroupDRO ResNet-50 fine-tune** or any 3rd/4th dataset until
the researcher signs off on these results (spec: cheap-first, then checkpoint).

If any arm was excluded (§2 gate) or CelebA was skipped, that is recorded above and in
`RESULTS_study.md` — report it honestly; do not silently substitute or re-tune.

In [ ]:
SKIPPED_DATASETS = sorted({d for _, d, _ in skipped})
if out["excluded"] or SKIPPED_DATASETS:
    lines = ["# BLOCKERS.md — grid arm exclusions / skips\n"]
    for e in out["excluded"]:
        lines.append(f"- EXCLUDED {e['backbone']}/{e['dataset']} {e['method']} seed{e['seed']}: "
                     f"worst-group acc {e['worst_group_acc']:.3f} < floor {e['floor']} ({e['reason']})")
    for d in SKIPPED_DATASETS:
        lines.append(f"- SKIPPED dataset {d}: not available in this run (see cell 4).")
    open("BLOCKERS.md", "w", encoding="utf-8").write("\n".join(lines) + "\n")
    print("wrote BLOCKERS.md")
print("\nPHASE = cheap-first last-layer grid COMPLETE. STOP for human review. "
      "Do NOT run the heavy full-GroupDRO fine-tune / 3rd-4th dataset yet.")